In [6]:
import pandas as pd

clean_df = pd.read_parquet(
    "../data/processed/medicines_cleaned.parquet"
)

print("Dataset shape:", clean_df.shape)
print("Pack size column exists:", "pack_size" in clean_df.columns)

Dataset shape: (253973, 17)
Pack size column exists: True


In [7]:
pack_counts = clean_df["pack_size"].value_counts(dropna=False)

print(pack_counts.head(30))

pack_size
strip of 10 tablets                116540
vial of 1 Injection                 17678
strip of 10 capsules                11931
strip of 6 tablets                   5718
bottle of 100 ml Syrup               5663
strip of 10 capsule sr               5511
strip of 10 tablet sr                5116
strip of 10 tablet dt                4309
vial of 2 ml Injection               3994
bottle of 30 ml Dry Syrup            3709
bottle of 60 ml Syrup                3330
strip of 15 tablets                  3260
vial of 1 ml Injection               2540
bottle of 60 ml Oral Suspension      2511
bottle of 30 ml Oral Suspension      2206
bottle of 30 ml Syrup                1896
strip of 10 tablet er                1871
strip of 3 tablets                   1796
strip of 10 tablet md                1778
tube of 15 gm Cream                  1763
strip of 1 Tablet                    1737
strip of 4 tablets                   1654
strip of 4 capsules                  1344
vial of 1 Powder for Inj

In [8]:
total_medicines = len(clean_df)

missing_pack_sizes = clean_df["pack_size"].isna().sum()

valid_pack_sizes = clean_df["pack_size"].notna().sum()

coverage = (valid_pack_sizes / total_medicines) * 100

print("Total medicines:", total_medicines)
print("Available pack sizes:", valid_pack_sizes)
print("Missing pack sizes:", missing_pack_sizes)
print(f"Pack-size coverage: {coverage:.2f}%")

Total medicines: 253973
Available pack sizes: 253973
Missing pack sizes: 0
Pack-size coverage: 100.00%


In [9]:
print(
    "Unique pack descriptions:",
    clean_df["pack_size"].nunique()
)

Unique pack descriptions: 1929


In [10]:

pack_units = ( clean_df["pack_size"].str.extract(r"\bof\s+(\d+(?:\.\d+)?)\s*(.*)$", expand=True))

pack_units.columns = ["quantity", "unit_description"]


unit_counts = (pack_units["unit_description"].value_counts(dropna=False)
)

print("Different quantity descriptions:")
print(unit_counts.head(50))

Different quantity descriptions:
unit_description
tablets                    133559
Injection                   18330
capsules                    14860
ml Syrup                    12894
ml Injection                12020
ml Oral Suspension           6347
tablet sr                    5637
capsule sr                   5604
gm Cream                     4994
tablet dt                    4634
ml Dry Syrup                 3841
ml Eye Drop                  3165
ml Suspension                2571
tablet er                    2155
tablet md                    1836
Tablet                       1748
gm Ointment                  1440
gm Gel                       1335
Powder for Injection         1291
Tablet pr                    1163
ml Lotion                    1014
soft gelatin capsules         903
ml Oral Drops                 888
ml Infusion                   819
tablet cr                     697
ml Oral Solution              568
ml Solution                   532
ml Ophthalmic Solution        52

In [11]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from pricing import add_pack_pricing

# Apply pack parsing and price normalization
clean_df = add_pack_pricing(clean_df)

print(clean_df[
    [
        "pack_size",
        "pack_quantity",
        "pack_unit",
        "pack_container",
        "pack_parse_status",
        "price",
        "price_per_unit"
    ]
].head(15))

                          pack_size  pack_quantity pack_unit pack_container  \
0               strip of 10 tablets           10.0    tablet          strip   
1                strip of 5 tablets            5.0    tablet          strip   
2            bottle of 100 ml Syrup          100.0        ml         bottle   
3               strip of 10 tablets           10.0    tablet          strip   
4               strip of 15 tablets           15.0    tablet          strip   
5               strip of 10 tablets           10.0    tablet          strip   
6               strip of 10 tablets           10.0    tablet          strip   
7                strip of 5 tablets            5.0    tablet          strip   
8               strip of 15 tablets           15.0    tablet          strip   
9            bottle of 100 ml Syrup          100.0        ml         bottle   
10              strip of 30 tablets           30.0    tablet          strip   
11           bottle of 100 ml Syrup          100.0  

In [12]:
total = len(clean_df)

parsed = clean_df["pack_parse_status"].eq("parsed").sum()

needs_review = clean_df["pack_parse_status"].eq(
    "needs_review"
).sum()

missing = clean_df["pack_parse_status"].eq("missing").sum()

valid_prices = clean_df["price_per_unit"].notna().sum()

print("Total records:", total)
print("Successfully parsed:", parsed)
print("Needs review:", needs_review)
print("Missing pack labels:", missing)

print(
    "Parser coverage:",
    round(parsed / total * 100, 2),
    "%"
)

print(
    "Valid price-per-unit records:",
    valid_prices
)

print(
    "Price-per-unit coverage:",
    round(valid_prices / total * 100, 2),
    "%"
)

Total records: 253973
Successfully parsed: 253079
Needs review: 894
Missing pack labels: 0
Parser coverage: 99.65 %
Valid price-per-unit records: 253075
Price-per-unit coverage: 99.65 %


In [13]:
review_labels = (
    clean_df.loc[
        clean_df["pack_parse_status"] == "needs_review",
        "pack_size"
    ]
    .value_counts()
)

print("Unparsed pack descriptions:")
print(review_labels.head(50))

Unparsed pack descriptions:
pack_size
packet of 5 suppositories             49
packet of 1 Inhaler                   47
packet of 5 respules                  44
packet of 30 rotacaps                 40
packet of 5 injections                32
bottle of 30 transcaps                20
vial of 10 injections                 18
strip of 1 Disintegrating Strip       16
box of 5 respules                     15
packet of 7 respules                  15
packet of 10 injections               15
box of 10 disintegrating strips       15
strip of 30 respicaps                 15
strip of 3 Vaginal Suppository        15
vial of 5 injections                  14
packet of 1 Sachet                    13
strip of 10 lozenges                  11
packet of 30 respicaps                11
ampoule of 5 injections               10
packet of 1 Transdermal Patch         10
bottle of 30 rotacaps                  9
box of 2 injections                    9
strip of 5 suppositories               9
packet of 1 Disinte

In [14]:
priced = clean_df.dropna(
    subset=["price_per_unit"]
).copy()

extremes = []

for unit, group in priced.groupby("pack_unit"):

    highest = group.nlargest(20, "price_per_unit").copy()
    highest["extreme_type"] = "highest"

    lowest = group.nsmallest(20, "price_per_unit").copy()
    lowest["extreme_type"] = "lowest"

    extremes.append(highest)
    extremes.append(lowest)

extremes_df = pd.concat(extremes, ignore_index=True)

print(
    extremes_df[
        [
            "name",
            "pack_size",
            "price",
            "pack_quantity",
            "pack_unit",
            "price_per_unit",
            "extreme_type"
        ]
    ].head(40).to_string(index=False)
)

                              name                         pack_size   price  pack_quantity pack_unit  price_per_unit extreme_type
               Zolmist Nasal Spray       bottle of 7 MDI Nasal Spray  650.05            7.0 actuation       92.864286      highest
               Zolvent Nasal Spray  pump bottle of 7 MDI Nasal Spray  537.00            7.0 actuation       76.714286      highest
             Calcisome Nasal Spray   bottle of 30 MDI Nasal Solution 1899.00           30.0 actuation       63.300000      highest
          Nasoshine AF Nasal Spray       bottle of 7 MDI Nasal Spray  349.00            7.0 actuation       49.857143      highest
               Citonin Nasal Spray   bottle of 30 MDI Nasal Solution 1450.00           30.0 actuation       48.333333      highest
               Calsalm Nasal Spray pump bottle of 60 MDI Nasal Spray 1990.00           60.0 actuation       33.166667      highest
         Medistat 1.25 Nasal Spray      bottle of 25 MDI Nasal Spray  638.88       

In [15]:
extremes_df.to_csv(
    "../data/processed/pack_price_extremes.csv",
    index=False
)

In [16]:
unit_summary = (
    clean_df.dropna(subset=["price_per_unit"])
    .groupby("pack_unit")
    .agg(
        medicine_count=("medicine_id", "count"),
        median_price_per_unit=("price_per_unit", "median"),
        minimum_price_per_unit=("price_per_unit", "min"),
        maximum_price_per_unit=("price_per_unit", "max")
    )
    .sort_values("medicine_count", ascending=False)
)

print(unit_summary)

                   medicine_count  median_price_per_unit  \
pack_unit                                                  
tablet                     152566               8.000000   
ml                          48558               2.066667   
capsule                     22111              10.450000   
vial                        18999             125.000000   
g                            9535               5.666667   
injection_unit                494             115.000000   
actuation                     305               3.008333   
kit                           246             107.500000   
ampoule                       117              90.000000   
prefilled_syringe              72            2074.500000   
patch                          47             300.000000   
pessary                        20               9.875000   
cartridge                       5            8000.000000   

                   minimum_price_per_unit  maximum_price_per_unit  
pack_unit                      

In [17]:
output_path = Path("../data/processed/medicines_phase3.parquet")

clean_df.to_parquet(output_path, index=False)

print("Phase 3 dataset saved!")
print("Location:", output_path.resolve())
print("Shape:", clean_df.shape)

Phase 3 dataset saved!
Location: D:\Engsci_Study_materials\Subject_Index\Projects\cheaper_medicine_finder\data\processed\medicines_phase3.parquet
Shape: (253973, 22)


In [18]:
# Inspect the 5 highest and 5 lowest values for EACH unit

priced = clean_df.dropna(
    subset=["price_per_unit"]
)

for unit, group in priced.groupby("pack_unit"):

    print("\n" + "=" * 70)
    print("UNIT:", unit)
    print("=" * 70)

    columns = [
        "name",
        "pack_size",
        "price",
        "price_per_unit"
    ]

    print("\nTOP 5 HIGHEST:")
    print(
        group.nlargest(5, "price_per_unit")[columns]
        .to_string(index=False)
    )

    print("\nTOP 5 LOWEST:")
    print(
        group.nsmallest(5, "price_per_unit")[columns]
        .to_string(index=False)
    )


UNIT: actuation

TOP 5 HIGHEST:
                    name                        pack_size   price  price_per_unit
     Zolmist Nasal Spray      bottle of 7 MDI Nasal Spray  650.05       92.864286
     Zolvent Nasal Spray pump bottle of 7 MDI Nasal Spray  537.00       76.714286
   Calcisome Nasal Spray  bottle of 30 MDI Nasal Solution 1899.00       63.300000
Nasoshine AF Nasal Spray      bottle of 7 MDI Nasal Spray  349.00       49.857143
     Citonin Nasal Spray  bottle of 30 MDI Nasal Solution 1450.00       48.333333

TOP 5 LOWEST:
                         name                     pack_size  price  price_per_unit
              AEROTAZ INHALER     packet of 200 MDI Inhaler  73.26        0.366300
             Beclate aquanase packet of 150 MDI Nasal Spray  55.25        0.368333
   Ventorlin Inhaler CFC Free     packet of 200 MDI Inhaler  80.64        0.403200
Aerocort 100mcg/50mcg Inhaler     packet of 400 MDI Inhaler 170.00        0.425000
     Asthavent 100mcg Inhaler        box of 2

In [19]:
mdi_records = clean_df[
    clean_df["pack_size"].str.contains(
        r"\bMDI\b",
        case=False,
        na=False
    )
]

print("Total MDI records:", len(mdi_records))

print(
    mdi_records[
        [
            "name",
            "pack_size",
            "pack_quantity",
            "pack_unit",
            "price",
            "price_per_unit"
        ]
    ].sample(
        min(20, len(mdi_records)),
        random_state=42
    ).to_string(index=False)
)

Total MDI records: 306
                                name                         pack_size  pack_quantity pack_unit  price  price_per_unit
                Isomet-F Nasal Spray     packet of 120 MDI Nasal Spray          120.0 actuation 249.00        2.075000
              Formonide 200 Novocart        packet of 100 MDI Novocart          100.0 actuation 395.00        3.950000
            Forglyn CFC Free Inhaler            box of 120 MDI Inhaler          120.0 actuation 544.50        4.537500
                 Levair B Transhaler      packet of 200 MDI Transhaler          200.0 actuation 125.00        0.625000
               Combitide 125 Inhaler         packet of 120 MDI Inhaler          120.0 actuation 545.00        4.541667
                 Azecool Nasal Spray pump bottle of 70 MDI Nasal Spray           70.0 actuation 395.00        5.642857
        Flomist AZ 50mcg Nasal Spray     packet of 100 MDI Nasal Spray          100.0 actuation 264.00        2.640000
        Fullform 200mcg/6

In [20]:
review_counts = (
    clean_df.loc[
        clean_df["pack_parse_status"] == "needs_review",
        "pack_size"
    ]
    .value_counts()
)

print("Unparsed unique descriptions:", len(review_counts))
print("Total unparsed records:", review_counts.sum())

print("\nTop 100 unparsed descriptions:")
print(review_counts.head(100).to_string())

Unparsed unique descriptions: 255
Total unparsed records: 894

Top 100 unparsed descriptions:
pack_size
packet of 5 suppositories             49
packet of 1 Inhaler                   47
packet of 5 respules                  44
packet of 30 rotacaps                 40
packet of 5 injections                32
bottle of 30 transcaps                20
vial of 10 injections                 18
strip of 1 Disintegrating Strip       16
box of 5 respules                     15
packet of 7 respules                  15
packet of 10 injections               15
box of 10 disintegrating strips       15
strip of 30 respicaps                 15
strip of 3 Vaginal Suppository        15
vial of 5 injections                  14
packet of 1 Sachet                    13
strip of 10 lozenges                  11
packet of 30 respicaps                11
ampoule of 5 injections               10
packet of 1 Transdermal Patch         10
bottle of 30 rotacaps                  9
box of 2 injections                

In [21]:
# Inspect injection pack descriptions containing measured quantities

injection_records = clean_df[
    clean_df["pack_size"].str.contains(
        r"injection",
        case=False,
        na=False
    )
]

injection_patterns = (
    injection_records["pack_size"]
    .value_counts()
)

print("Total injection records:", len(injection_records))

print("\nInjection pack descriptions:")
print(injection_patterns.head(60).to_string())

Total injection records: 32044

Injection pack descriptions:
pack_size
vial of 1 Injection                                     17678
vial of 2 ml Injection                                   3994
vial of 1 ml Injection                                   2540
vial of 1 Powder for Injection                           1283
vial of 5 ml Injection                                   1036
vial of 10 ml Injection                                   873
ampoule of 2 ml Injection                                 688
ampoule of 1 ml Injection                                 595
box of 1 Injection                                        400
vial of 3 ml Injection                                    314
vial of 20 ml Injection                                   244
ampoule of 5 ml Injection                                 185
vial of 30 ml Injection                                   184
vial of 4 ml Injection                                    181
ampoule of 1 Injection                                    116

In [ ]:
# Find injection labels with measurement units

measured_injections = injection_records[
    injection_records["pack_size"].str.contains(
       r"\b(mg|mcg|gm|g|iu|ml|ltr)\b",
        case=False,
        na=False
    )
]

print(
    measured_injections[
        ["name", "pack_size", "price"]
    ].head(50).to_string(index=False)
)

                                    name                                     pack_size   price
                          Avil Injection                       vial of 10 ml Injection   21.17
       Amikacin Sulphate 500mg Injection                        vial of 2 ml Injection   11.87
                          Avil Injection                        vial of 2 ml Injection    5.66
           Adrenaline Tartrate Injection                        vial of 1 ml Injection   14.52
             Atropine Sulphate Injection                       vial of 10 ml Injection   11.25
            Actrapid HM 100IU/ml Penfill        penfill of 3 ml Solution for Injection  350.90
                  Apidra 100IU Cartridge      cartridge of 3 ml Solution for Injection  761.00
      Alburel 20gm Solution for Infusion                    bottle of 100 ml Injection 7325.23
                        Aciloc Injection                        vial of 2 ml Injection    6.03
              Actrapid 100 IU/ml Flexpen        fl

C:\Users\HP\AppData\Local\Temp\ipykernel_30588\386453574.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  injection_records["pack_size"].str.contains(
